# Dezrann's structure and texture of Mozart's sonatas, added to the DCML files

Download Dezrann's archive (md5 checked), build the CSVs and the ODbL package, merge the layers into DCML's files for the nine movements of K. 279, 280 and 283, run TiLiA, print the counts. Output directory: `$DATASET_CONVERTERS_OUT`; DCML's converted files are read from `$DATASET_CONVERTERS_OUT/../dcml` unless `DATASET_CONVERTERS_DCML` says otherwise (they are made by `dcml-mozart/convert.py` when missing). Needs `TILIA` pointing at TiLiA's command line. The merged files are built locally and never shipped.

In [ ]:
import importlib.util, os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "common").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

dez = load("dezrann_convert", root / "dezrann-mozart" / "convert.py")
out = Path(os.environ["DATASET_CONVERTERS_OUT"])
out.mkdir(parents=True, exist_ok=True)
dcml_dir = Path(os.environ.get("DATASET_CONVERTERS_DCML", out.parent / "dcml"))

## DCML's files

The converted DCML movements are read, never changed; any that is missing is converted first.

In [ ]:
missing = [p for p in dez.PIECES if not (dcml_dir / "tla" / f"{p}.tla").exists()]
if missing:
    dcml = load("dcml_convert", root / "dcml-mozart" / "convert.py")
    dcml.convert(missing, dcml_dir)
print(len(dez.PIECES) - len(missing), "DCML files found,", len(missing), "converted")

## Download

One archive, 1.6 MB, DOI 10.57745/OHRWPC (v1.0); `fetch` rejects a wrong md5.

In [ ]:
archive = dez.fetch_archive()
print(archive, dez.MD5)

## Build the CSVs, merge, run TiLiA

CSVs by bar (`csv-by-bar/`, what the package ships) and by time in seconds (`csv-by-time/`, what is imported). The merged files go to `tla/`; TiLiA opens a copy of each DCML file.

In [ ]:
summary = dez.convert(dez.PIECES, dcml_dir, out, archive=archive)

## Package

Dezrann's layers alone, as CSVs by bar, under the ODbL. The merged `.tla` files are not packaged: they hold DCML's CC BY-NC-SA timelines next to Dezrann's ODbL ones.

In [ ]:
print(dez.package(out, out / "package"))

## Bar numbers

Dezrann's measure maps against DCML's Measures timelines: bar numbers and bar start times (voltas count once each, in order, in both).

In [ ]:
for piece, s in summary.items():
    b = s["bar_numbers"]
    print(f"{piece}: {b['bars']:3} bars  numbers agree {b['numbers_agree']}  starts agree {b['starts_agree']}  "
          f"pickup {b['pickup']}  repeated numbers {b['repeated_numbers']}")
assert all(s["bar_numbers"]["numbers_agree"] and s["bar_numbers"]["starts_agree"] for s in summary.values())

## Counts

Components in the merged files against the labels of each type in the `.dez` files.

In [ ]:
tot = {"Structure": [0, 0], "Texture": [0, 0]}
for piece, s in summary.items():
    for typ in tot:
        tot[typ][0] += s["components"][typ]; tot[typ][1] += s["dez_labels"][typ]
    print(f"{piece}: structure {s['components']['Structure']:3}/{s['dez_labels']['Structure']:3}  "
          f"texture {s['components']['Texture']:3}/{s['dez_labels']['Texture']:3}  DCML unchanged {s['dcml_unchanged']}")
for typ, (comps, labels) in tot.items():
    print(f"{typ:10} components {comps:5}  .dez labels {labels:5}")
assert all(c == l for c, l in tot.values()) and all(s["dcml_unchanged"] for s in summary.values())